# Interpretation experiment for Advection
This notebook creates the small mini-figures for the interpretation results for advection

In [ ]:
import os
import sys
from pathlib import Path

# Force notebook CWD to project root: .../pde_lightning
PROJECT_ROOT = Path.cwd().resolve().parent  # from notebooks/ -> parent
os.chdir(PROJECT_ROOT)

# Keep imports stable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("CWD:", Path.cwd())

print("PROJECT_ROOT:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())


In [ ]:
import json
from pathlib import Path
import torch
import pytorch_lightning as pl

from src.train import build_configs
from src.data.equation_datamodule import EquationModule
from src.models.base_lightning_model import PDELightningModule

# Target run
run_dir = Path(
    "outputs/advection_benchmark/"
    "advection_benchmark_late_fusion_sparsity_sweep_sparsity_weight-1e-4_seed234"
)

# Load resolved config written during training
resolved = json.loads((run_dir / "resolved_config.json").read_text(encoding="utf-8"))

equation = resolved["equation"]
model_name = resolved["model"]
training = resolved.get("training", {})

# Rebuild configs exactly like training
resolved_model_name, model_config, datamodule_config = build_configs(equation, model_name)
model_config.update(resolved.get("model_config", {}))
datamodule_config.update(resolved.get("datamodule_config", {}))

# DataModule + test loaders
dm = EquationModule(**datamodule_config)
dm.setup(stage=None)
dm.batch_size_test = 500  # optional override

test_id_loader = dm.test_dataloader()

# Pick checkpoint (prefer last.ckpt, fallback to best epoch ckpt)
ckpt_dir = run_dir / "checkpoints"
ckpt_path = ckpt_dir / "last.ckpt"
if not ckpt_path.exists():
    best_ckpts = sorted(ckpt_dir.glob("epoch=*.ckpt"))
    if not best_ckpts:
        raise FileNotFoundError(f"No checkpoint found in: {ckpt_dir}")
    ckpt_path = best_ckpts[-1]

# Load model
model = PDELightningModule.load_from_checkpoint(
    str(ckpt_path),
    model_name=resolved_model_name,
    model_config=model_config,
    learning_rate=float(training.get("learning_rate", 1e-3)),
    log_hyperparameters=False,
    strict=False,
)

# Move to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()

In [ ]:
# -----------------------------
# 1) Get one batch and time step
# -----------------------------
batch = next(iter(test_id_loader))
batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

state = batch["state"]  # expected: [B, T, X, V]
B, T, X, V = state.shape

t = batch.get("t", None)
if t is not None:
    t_cpu = t.detach().cpu()
    # handle [T], [B,T], [B,T,1], etc.
    if t_cpu.ndim == 1:
        t_vec = t_cpu
    else:
        t_vec = t_cpu[0].reshape(-1)
    dt = float((t_vec[1] - t_vec[0]).item()) if t_vec.numel() > 1 else np.nan
else:
    t_vec = torch.arange(T, dtype=torch.float32)
    dt = 1.0  # fallback if no physical time coordinate provided

print(f"Batch shape [B,T,X,V]: {tuple(state.shape)}")
print(f"Estimated time step dt: {dt}")

In [ ]:
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

FIGSIZE = (1.2, 1.0)
FONTSIZE = 7
PLOT_RECT = [0.22, 0.18, 0.74, 0.76]
CBAR_RECT = [0.88, 0.18, 0.04, 0.76]
plt.rcParams.update({"font.size": FONTSIZE})

def _new_plot_axes():
    fig = plt.figure(figsize=FIGSIZE)
    ax = fig.add_axes(PLOT_RECT)
    ax.xaxis.set_major_locator(MaxNLocator(3))
    ax.yaxis.set_major_locator(MaxNLocator(3))
    ax.set_xticks([0, 1])
    ax.tick_params(labelsize=FONTSIZE, pad=0.5)
    return fig, ax

def _new_heatmap_axes():
    fig = plt.figure(figsize=FIGSIZE)
    ax = fig.add_axes(PLOT_RECT)
    cax = fig.add_axes(CBAR_RECT)
    ax.xaxis.set_major_locator(MaxNLocator(3))
    ax.yaxis.set_major_locator(MaxNLocator(3))
    ax.set_xticks([0, 1])
    ax.tick_params(labelsize=FONTSIZE, pad=0.5)
    cax.tick_params(labelsize=FONTSIZE, pad=0.5)
    return fig, ax, cax

id = 3
# ------------------------------------------
# 2) Plot first trajectory
# ------------------------------------------
u = state[id, 0, :, 0].detach().cpu().numpy()
x = batch["x"].detach().cpu().numpy()[0]

fig, ax = _new_plot_axes()
ax.plot(x, u, lw=1.0)

out_path = Path("outputs/interpretation/advection/input.svg")
out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_path, format="svg", bbox_inches="tight", pad_inches=0.01)
plt.show()
print(f"Saved: {out_path}")

# ------------------------------------------
# 3) Plot derivative using finite difference
# ------------------------------------------
du_dx = np.gradient(u, x)
d2u_dx2 = np.gradient(du_dx, x)
params = batch["parameter"][id].item()

fig, ax = _new_plot_axes()
ax.plot(x, -params*du_dx, lw=1.0)
out_path = Path("outputs/interpretation/advection/-dudx.svg")
out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_path, format="svg", bbox_inches="tight", pad_inches=0.01)
plt.show()
print(f"Saved: {out_path}")

In [ ]:
# ----------------------------------------------------
# 3) Hidden state from FNO for first input time slice
# ----------------------------------------------------
core = model.model
assert hasattr(core, "fno"), "Loaded model wrapper does not expose .fno"
assert hasattr(core, "library"), "Loaded model wrapper does not expose .library"
assert hasattr(core, "regression"), "Loaded model wrapper does not expose .regression"

with torch.no_grad():
    input_state_t0 = state[:, 0, :, :]   # [B, X, V]
    h = core.fno(input_state_t0, batch["x"])  # usually [B, X, 1, H]
    print(h.shape)
    if h.ndim == 4:
        h = h.squeeze(-2)                # -> [B, X, H]
    hidden_first = h[id].detach().cpu().numpy()  # [X, H]

print("Hidden state shape for first trajectory at t0:", hidden_first.shape)

for ch_idx in range(hidden_first.shape[1]):
    fig, ax_h = _new_plot_axes()
    ax_h.plot(x, hidden_first[:, ch_idx], lw=1.0)
    out_path = Path(f"outputs/interpretation/advection/hiddenstate_{ch_idx}.svg")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, format="svg", bbox_inches="tight", pad_inches=0.01)
    plt.show()
    print(f"Saved: {out_path}")

In [ ]:
W = core.regression.get_coefficients().detach().cpu()  # [output_dim, library_size]
print("Coefficient matrix shape:", tuple(W.shape))
print("Coefficient matrix:\n", W.numpy())

coef_0 = W[:, 0].item()  # example: first weight of first library term
coef_1 = W[:, 1].item()  # example: first weight of second library term
print(coef_0, coef_1)
params = batch["parameter"][id].item()  # example: parameter for this trajectory

hidden_state_scaled = hidden_first[:, 0] / dt * params * coef_0
hidden_state_2_scaled = hidden_first[:, 1] / dt * coef_1

fig, ax = _new_plot_axes()
ax.plot(x, hidden_state_scaled, lw=1.0)

# Store limits from first plot
xlim = ax.get_xlim()
ylim = ax.get_ylim()

out_path = Path("outputs/interpretation/advection/hiddenstate_0_scaled.svg")
out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_path, format ="svg",bbox_inches="tight", pad_inches=0.01)
plt.show()
print(f"Saved: {out_path}")

fig, ax = _new_plot_axes()
ax.plot(x, hidden_state_2_scaled, lw=1.0)

# Reuse limits from first plot
ax.set_xlim(xlim)
ax.set_ylim(ylim)

out_path = Path("outputs/interpretation/advection/hiddenstate_1_scaled.svg")
out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_path,format ="svg", bbox_inches="tight", pad_inches=0.01)
plt.show()
print(f"Saved: {out_path}")

fig, ax = _new_plot_axes()
ax.plot(x, u, lw=1.0)
out_path = Path("outputs/interpretation/advection/output.svg")
out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_path, format ="svg", bbox_inches="tight", pad_inches=0.01)
plt.show()
print(f"Saved: {out_path}")

In [ ]:
import pandas as pd

# -------------------------------------------------
# 4) Coefficient matrix + corresponding term names
# -------------------------------------------------
W = core.regression.get_coefficients().detach().cpu()  # [output_dim, library_size]
print("Coefficient matrix shape:", tuple(W.shape))
print("Coefficient matrix:\n", W.numpy())

hidden_dim = hidden_first.shape[1]
param_dim = batch["parameter"].shape[-1]
terms = core.library.get_feature_names(hidden_dim=hidden_dim, param_dim=param_dim)

if len(terms) != W.shape[1]:
    print(f"Warning: number of terms ({len(terms)}) != number of coefficients ({W.shape[1]})")

# Build readable table: one row per library term
coef_table = pd.DataFrame({"term": terms[:W.shape[1]]})
for out_i in range(W.shape[0]):
    coef_table[f"coef_out{out_i}"] = W[out_i, :len(coef_table)].numpy()

print("\nTerm -> coefficients:")
display(coef_table)

# Optional: show strongest terms (by abs coefficient) for first output
topk = min(20, len(coef_table))
top_terms = coef_table.assign(abs_coef=np.abs(coef_table["coef_out0"])).sort_values("abs_coef", ascending=False).head(topk)
print(f"\nTop {topk} terms by |coef_out0|:")
display(top_terms[["term", "coef_out0", "abs_coef"]])

In [ ]:
g = model(input_state_t0, batch["parameter"], batch["x"])
print("Model output shape:", h.shape)

# Put g on the cpu
g = g[id].detach().cpu().numpy()  # [X, output_dim]

fig, ax = _new_plot_axes()
ax.plot(x, g, lw=1.0)
plt.show()